# Chapter 04-08 · Reproducibility, seeds, and experiment records

**Label:** Core  |  **Time:** ~45 minutes  |  **Difficulty:** easy to read, easy to neglect

**Prerequisites:** 04-03 for the split lottery, 04-07 for pipelines.

**Position in the learning path:** module 04, chapter 8 of 8 - the last.

---

## Why this matters

Module 04 has built a workflow that is **correct**. This chapter makes it **repeatable**, which is a
different property and is missing far more often.

A result nobody can reproduce is not a weaker result. **It is not a result at all** - there is no way to
check it, extend it, or debug it when it stops working. And the usual response, "I set `random_state=42`",
fixes one of five things that have to be fixed, and not the most important one.

This chapter measures which of them actually move the number. The answer is not what people spend their
time on: **the split seed moves it five times more than the model seed does**, and neither is what breaks
a result six months later.

## What you will be able to do

- Say precisely what a seed does and does not control
- Measure how much of a reported number is seed choice, and which seed
- Recognise the sources of non-determinism a seed cannot touch
- Write an experiment record somebody else can act on
- Pin an environment, and explain why the pinning matters more than the seed

## Warm-up: retrieve, do not reread

1. In 04-03, how far apart were the best and worst random splits of the same data?
2. What does a `Pipeline` guarantee that a loose preprocessing step does not?
3. In 04-07, what turned out to explain the gap between `best_score_` and the nested score?

<br>

*Answers: (1) AUC 0.608 to 0.819 for the same model. (2) that every fitted step is refitted inside every
fold. (3) not selection - the inner loop trained on 3/4 of the rows instead of 4/5.*

## Level 1 · The same code, twice

Start with the failure that is easiest to fix and easiest to miss.

In [ ]:
import platform
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# SYNTHETIC: two informative columns, one irrelevant, and an interaction
rng = np.random.default_rng(11)
n_rows = 800
features = pd.DataFrame({"x1": rng.normal(size=n_rows),
                         "x2": rng.normal(size=n_rows),
                         "x3": rng.normal(size=n_rows)})
target = (2 * features.x1 - 1.5 * features.x2
          + 0.5 * features.x1 * features.x2 + rng.normal(0, 1.0, n_rows)).to_numpy()

train_rows, test_rows = train_test_split(np.arange(n_rows), test_size=0.3, random_state=0)


def fit_and_score(model_seed=None):
    forest = RandomForestRegressor(n_estimators=50, min_samples_leaf=3, random_state=model_seed)
    forest.fit(features.iloc[train_rows], target[train_rows])
    return mean_absolute_error(target[test_rows], forest.predict(features.iloc[test_rows]))


unseeded = [fit_and_score() for _ in range(6)]
seeded = [fit_and_score(model_seed=0) for _ in range(6)]

print("no random_state - six runs of identical code:")
print("  " + "  ".join("%.6f" % value for value in unseeded[:3]))
print("  " + "  ".join("%.6f" % value for value in unseeded[3:]))
print("  they span %.6f, which is %.1f%% of the score"
      % (max(unseeded) - min(unseeded), 100 * (max(unseeded) - min(unseeded)) / np.mean(unseeded)))
print()
print("random_state=0 - six runs of identical code:")
print("  " + "  ".join("%.6f" % value for value in seeded[:3]))
print("  they span %.1e - every run is the same number" % (max(seeded) - min(seeded)))

**Without a seed, the same code on the same data gives a different answer every time** - the six runs
above span a percent or two, and they will span a different amount when you re-run this notebook, which is
the point. With a seed, every run is the same number to the last decimal, forever.

That is worth doing, and it is where most people stop. **It is also the least important of the five
levels**, because it only guarantees that *you* get *your* number back on *your* machine today.

## The five levels of reproducibility

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.6))
levels = [
    ("1  Seeds", "#cfe3f3", "`random_state` on every estimator, splitter and generator",
     "same answer twice on this machine, today"),
    ("2  Code", "#cfe3f3", "a git commit, not 'the notebook as I left it'",
     "somebody can run the same steps"),
    ("3  Data", "#f6d3bd", "a snapshot or a query with a date, not 'the table'",
     "the same rows go in"),
    ("4  Environment", "#f6d3bd", "pinned versions of python and every library",
     "the same arithmetic comes out"),
    ("5  Hardware", "#f3d6e3", "thread count, BLAS, GPU, float precision",
     "the last few decimals agree"),
]
for position, (title, colour, what, buys) in enumerate(levels):
    y = len(levels) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.08), 2.3, 0.84, facecolor=colour, edgecolor="white",
                               linewidth=2.5))
    ax.text(1.2, y + 0.5, title, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(2.55, y + 0.63, what, va="center", fontsize=10, color="#222222")
    ax.text(2.55, y + 0.3, "buys you: " + buys, va="center", fontsize=8.5, color="#777777",
            style="italic")
ax.set_xlim(0, 11.6)
ax.set_ylim(-0.15, len(levels) + 0.35)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Each level fixes something the one above it cannot", fontsize=13)
plt.tight_layout()
plt.show()

**A seed pins level 1 and nothing else.** If the library version changes, the seed produces a different
sequence; if the data has been updated in place, the rows differ; if the notebook has been edited, the
code differs. Every one of those is more likely to happen over six months than a missing `random_state`.

The rest of this chapter takes them in order of how much they actually cost you.

## Which seed actually matters?

There are two seeds in the code above - one on the splitter and one on the forest - and they are usually
treated as interchangeable housekeeping.

**Predict before running:** vary each one over thirty values, holding the other fixed. Which produces more
variation in the reported score?

In [ ]:
def score(split_seed, model_seed):
    train_index, test_index = train_test_split(np.arange(n_rows), test_size=0.3,
                                               random_state=split_seed)
    forest = RandomForestRegressor(n_estimators=50, min_samples_leaf=3, random_state=model_seed)
    forest.fit(features.iloc[train_index], target[train_index])
    return mean_absolute_error(target[test_index], forest.predict(features.iloc[test_index]))


split_varies = np.array([score(seed, 0) for seed in range(30)])
model_varies = np.array([score(0, seed) for seed in range(30)])
both_vary = np.array([score(seed, seed) for seed in range(30)])

summary = pd.DataFrame([
    {"what varies": "the split seed", "mean": round(split_varies.mean(), 4),
     "sd": round(split_varies.std(), 4),
     "range": "%.4f - %.4f" % (split_varies.min(), split_varies.max())},
    {"what varies": "the model seed", "mean": round(model_varies.mean(), 4),
     "sd": round(model_varies.std(), 4),
     "range": "%.4f - %.4f" % (model_varies.min(), model_varies.max())},
    {"what varies": "both", "mean": round(both_vary.mean(), 4),
     "sd": round(both_vary.std(), 4),
     "range": "%.4f - %.4f" % (both_vary.min(), both_vary.max())},
])
print(summary.to_string(index=False))
print()
print("the split seed produces %.1f times the spread of the model seed"
      % (split_varies.std() / model_varies.std()))

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.4))

bins = np.linspace(0.75, 1.06, 34)
left.hist(split_varies, bins=bins, alpha=0.8, color="#D55E00", label="split seed varies")
left.hist(model_varies, bins=bins, alpha=0.8, color="#0072B2", label="model seed varies")
left.set_xlabel("MAE")
left.set_ylabel("runs")
left.set_title("Thirty seeds each. One of them barely moves", fontsize=11)
left.legend(fontsize=8)

right.bar([0, 1, 2], [split_varies.std(), model_varies.std(), both_vary.std()],
          color=["#D55E00", "#0072B2", "#999999"], width=0.6)
for position, value in enumerate([split_varies.std(), model_varies.std(), both_vary.std()]):
    right.text(position, value + 0.0012, "%.4f" % value, ha="center", fontsize=10)
right.set_xticks([0, 1, 2])
right.set_xticklabels(["split seed", "model seed", "both"], fontsize=9)
right.set_ylabel("standard deviation of the reported MAE")
right.set_title("'Both' is indistinguishable from 'split alone'", fontsize=11)

plt.tight_layout()
plt.show()

**The split seed produces 5.2 times the spread of the model seed** - sd 0.0487 against 0.0094 - and
varying both gives 0.0481, which is the split's contribution with the model's rounding error on top.

Two things follow.

**Reporting a number from one split is the thing to worry about, not the model's internal randomness.**
This is 04-03's split lottery, arriving as a reproducibility question instead of an evaluation one. If you
want a number that does not move, the answer is not a better seed - it is **cross-validation, and
reporting the spread**.

**And notice the means differ too**: 0.8809 when the split varies against 1.0032 when it is pinned at
`random_state=0`. Split 0 happens to be a hard one. Somebody who wrote `random_state=0` once, at the
start, has been reporting a number about 14% worse than the average split all along - through no fault
and with no warning.

> **A seed makes a number stable. It does not make it representative.** Those are different problems, and
> stability is the easier one.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.3))

left.plot(range(30), split_varies, "o-", color="#D55E00", markersize=5, linewidth=1)
left.axhline(split_varies.mean(), color="#000000", linestyle="--", linewidth=1.4)
left.plot([0], [split_varies[0]], "o", color="#0072B2", markersize=13)
left.annotate("random_state=0\nreports %.4f" % split_varies[0], (0, split_varies[0]),
              textcoords="offset points", xytext=(18, -8), color="#0072B2", fontsize=9)
left.text(29, split_varies.mean() + 0.004, "average over seeds: %.4f" % split_varies.mean(),
          ha="right", fontsize=9)
left.set_xlabel("split seed")
left.set_ylabel("reported MAE")
left.set_title("Thirty equally valid seeds. Somebody picked the first one", fontsize=11)

edges = np.linspace(0.77, 1.03, 40)
right.hist(seeded, bins=edges, color="#0072B2", label="one seed, six runs")
right.hist(split_varies, bins=edges, alpha=0.7, color="#D55E00", label="thirty seeds, one run each")
right.set_xlabel("reported MAE")
right.set_ylabel("runs")
right.set_title("Stable is not the same as representative", fontsize=11)
right.legend(fontsize=8)

plt.tight_layout()
plt.show()

The left panel is thirty equally defensible choices, and the one somebody happened to type. The right
panel is the distinction this chapter turns on: **a seed collapses the blue spike to a single point, and
says nothing at all about where that point sits inside the orange distribution.**

## What a seed cannot fix

Below the seed there is arithmetic, and arithmetic on a computer is not quite the arithmetic you learned.

In [ ]:
values = rng.normal(size=100_000) * 1e6
reordered = values.copy()
np.random.default_rng(1).shuffle(reordered)

first = values.sum()
second = reordered.sum()
print("the same 100,000 numbers, added in two different orders:")
print("  %.10f" % first)
print("  %.10f" % second)
print("  difference: %.3e  (relative: %.2e)" % (abs(first - second), abs(first - second) / abs(first)))

**The same numbers, added in a different order, give a different total.** Floating-point addition is not
associative - `(a + b) + c` and `a + (b + c)` can differ in the last bit - so any operation whose order
depends on how work was divided between threads is not bit-reproducible.

The difference here is about **one part in 10^15**, which is the last bit of a float64 and is almost never
a problem on its own. It matters because it **compounds**: a tie broken differently in a tree split, a
different first centroid in k-means, a slightly different gradient step, and the paths diverge.

The practical list of things a seed does not control:

| Source | Why a seed does not help | What to do |
|---|---|---|
| **Thread count** (`n_jobs`, BLAS) | changes the order of summation | pin it, or accept last-decimal differences |
| **Library versions** | the algorithm itself may change | pin versions - the highest-value item on this list |
| **GPU** | massively parallel reduction, non-deterministic by default | use the framework's deterministic mode, accept the cost |
| **Python hash randomisation** | set and dict iteration order varies between processes | sort before iterating; never rely on set order |
| **Data updated in place** | the rows are different | snapshot, or query with an as-of date |

**Of those five, library versions are the one that will actually break you**, and it will happen when
somebody installs the project a year later and gets scikit-learn 1.12 instead of 1.9. A default changed, a
solver was replaced, and the number is different with no error and no clue.

In [ ]:
import sklearn

record = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "platform": "%s %s" % (platform.system(), platform.machine()),
}
print("the minimum environment record, which costs five lines:")
for key, value in record.items():
    print("  %-14s %s" % (key, value))

Five lines, captured automatically, and they turn "it gives a different answer now" from a mystery into a
diff.

Which lets the chapter's sources of variation be put on one axis - with the important caveat that one of
them cannot be measured at all.

In [ ]:
relative = [
    ("choice of split seed", split_varies.std() / split_varies.mean(), "#D55E00"),
    ("choice of model seed", model_varies.std() / model_varies.mean(), "#0072B2"),
    ("float summation order", abs(first - second) / abs(first), "#999999"),
]

fig, ax = plt.subplots(figsize=(9.5, 4.2))
for position, (label, size, colour) in enumerate(relative):
    ax.barh(position, size, color=colour, height=0.55)
    ax.text(size * 1.5, position, "%.1e" % size, va="center", fontsize=9.5)
ax.barh(3, 1.0, color="#f3d6e3", height=0.55)
ax.text(1.5e-14, 3, "a library upgrade: no measurable size - it can change a default,"
        + chr(10) + "a solver, or nothing at all, and you find out by accident",
        va="center", fontsize=9, color="#8a4a68")

ax.set_yticks(range(4))
ax.set_yticklabels(["split seed", "model seed", "float order", "library version"], fontsize=10)
ax.set_xscale("log")
ax.set_xlim(1e-16, 3e2)
ax.set_xlabel("relative size of the change it can make to your reported number (log scale)")
ax.set_title("Thirteen orders of magnitude, and the biggest one has no bar", fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**The split seed moves the number about 5% of its own size. Float ordering moves it by one part in a
thousand million million.** Those two are thirteen orders of magnitude apart, and people worry about them
in roughly the opposite proportion.

The pink bar is drawn full width because **it has no measurable size**. A library upgrade might change
nothing, or might change a default from `auto` to something else and move your result by more than
everything above it combined. You cannot bound it in advance, which is exactly why pinning versions is the
highest-value line in this chapter - and why it is level 4 rather than level 1.

**This is what `requirements.txt` with pinned versions is for**, and why this course's has them. A
requirements file listing `scikit-learn` without a version documents an intention, not an environment.

## The experiment record

Everything above is about getting the *same* number twice. An experiment record is about knowing **which
number it was, and why you ran it** - and it is what turns a folder of notebooks into work somebody can
build on.

The test of a record is simple: **could a competent stranger, or you in six months, reproduce this result
and say what it means?** That requires more than the code.

In [ ]:
import json
import subprocess


def experiment_record(name, notes, scores, settings):
    try:
        commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        commit = "not a git repository"
    return {
        "experiment": name,
        "why": notes,
        "code": commit or "uncommitted",
        "data": "synthetic, generator seed 11, 800 rows",
        "split": "train_test_split(test_size=0.3, random_state=0)",
        "settings": settings,
        "result": {"metric": "MAE", "value": round(float(np.mean(scores)), 4),
                   "spread": round(float(np.std(scores)), 4), "runs": len(scores)},
        "baseline": {"metric": "MAE", "value": round(float(np.mean(np.abs(
            target[test_rows] - np.median(target[train_rows])))), 4),
            "what": "predict the training median"},
        "environment": record,
    }


entry = experiment_record(
    name="forest-vs-median-baseline",
    notes="does a random forest beat the constant baseline on the synthetic interaction data?",
    scores=split_varies,
    settings={"model": "RandomForestRegressor", "n_estimators": 50, "min_samples_leaf": 3,
              "random_state": 0})
print(json.dumps(entry, indent=2))

Read what is in there that a notebook alone does not have.

**The baseline**, from 04-02, so the number means something. **The spread**, from 04-03, so the reader
knows how much of it is noise. **The split rule**, from 04-04, so they know what was held out. **The data
identity**, so they know the rows. **The commit**, so they can get the code. **The environment**, so the
arithmetic matches.

And **`why`** - a sentence saying what question this run was supposed to answer. That is the field people
omit and the one that makes a folder of forty runs navigable a month later.

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.4))
fields = [
    ("why", "the question this run answers", "#f3d6e3", "the field people omit"),
    ("code", "a commit hash", "#cfe3f3", "not 'the notebook'"),
    ("data", "a snapshot id or an as-of date", "#cfe3f3", "not 'the table'"),
    ("split", "the exact splitting rule", "#f6d3bd", "04-04"),
    ("settings", "every hyperparameter, including the seeds", "#f6d3bd", "04-07"),
    ("result", "the metric, its value, and its spread", "#cfe8dc", "04-03"),
    ("baseline", "what doing nothing scores", "#cfe8dc", "04-02"),
    ("environment", "python and library versions", "#cfe3f3", "the one that breaks"),
]
for position, (field, meaning, colour, note) in enumerate(fields):
    y = len(fields) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.08), 2.0, 0.84, facecolor=colour, edgecolor="white",
                               linewidth=2))
    ax.text(1.05, y + 0.5, field, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(2.25, y + 0.5, meaning, va="center", fontsize=10, color="#222222")
    ax.text(8.6, y + 0.5, note, va="center", fontsize=8.5, color="#888888", style="italic")
ax.set_xlim(0, 11.4)
ax.set_ylim(-0.15, len(fields) + 0.35)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("An experiment record, and where each field came from in this module", fontsize=13)
plt.tight_layout()
plt.show()

**Every field is something a previous chapter of module 04 argued for.** The record is not extra
bureaucracy bolted onto the workflow - it is the workflow, written down.

## When the number changes: a diagnosis order

The record earns its keep on the day a result stops reproducing. Work down the levels, cheapest first.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.9))
steps = [
    ("Did the environment change?", "diff the version record. The commonest cause, and a one-line check"),
    ("Did the data change?", "row count, date range, checksum. Tables get updated in place"),
    ("Did the code change?", "diff the commit. 'I only cleaned it up' is a code change"),
    ("Is a seed missing?", "grep for estimators and splitters without `random_state`"),
    ("Is it just the last decimals?", "then it is thread count or float order, and it does not matter"),
]
for position, (question, detail) in enumerate(steps):
    y = len(steps) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.1), 9.9, 0.8,
                               facecolor="#cfe3f3" if position < 3 else "#eeeeee",
                               edgecolor="white", linewidth=2.5))
    ax.text(0.3, y + 0.63, question, va="center", fontsize=11, fontweight="bold")
    ax.text(0.3, y + 0.3, detail, va="center", fontsize=9, color="#555555")
    if position < len(steps) - 1:
        ax.annotate("", xy=(5.0, y + 0.04), xytext=(5.0, y + 0.1),
                    arrowprops=dict(arrowstyle="-|>", color="#888888", linewidth=1.6))
ax.set_xlim(0, 10.2)
ax.set_ylim(-0.1, len(steps) + 0.4)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("When a result stops reproducing, in the order that costs least", fontsize=13)
plt.tight_layout()
plt.show()

**Notice that "is a seed missing?" is fourth.** It is the first thing people check and the least likely
cause, because a missing seed produces *small, noisy* differences while the three above it produce large
or structural ones. Matching the size of the discrepancy to the level is most of the diagnosis: a change
in the third decimal is arithmetic, a change in the first is data or code.

## Common misconceptions

**"`random_state=42` makes it reproducible."**
It makes it repeatable on your machine, with today's libraries. It is level 1 of five.

**"The seed choice does not matter, it is arbitrary."**
Arbitrary and consequential are not opposites. `random_state=0` here reports 1.0032 where the average
split gives 0.8809 - 14% pessimistic, permanently, by accident.

**"Reproducible means identical to the last decimal."**
Bit-identical is a strong and often expensive requirement. Usually what you need is that the *conclusion*
reproduces, and last-decimal differences from thread ordering are irrelevant.

**"I'll write up the experiment record at the end."**
The record is generated by the code that runs the experiment, or it is written from memory and wrong.

**"The notebook is the record."**
A notebook records what you last ran, not what you ran when you got the number you reported - and not the
data version or the library versions.

**"Pinning versions is over-engineering for a small project."**
It is one file and it is the level that actually breaks. Small projects are exactly the ones nobody
revisits for a year.

## Exercises

Solutions: `solutions/04_workflow/04-08_reproducibility_solutions.ipynb`.

### Quick understanding

**E1.** Name the five levels of reproducibility and what each one fixes.

**E2.** Which seed in this chapter mattered more, by how much, and why?

**E3.** Give two sources of non-determinism that a seed cannot control.

### Hand calculation

**E4.** A reported MAE is 0.8809 with a split-to-split standard deviation of 0.0487. Somebody reports
0.7859 from their favourite seed. How many standard deviations below the mean is that, and what would you
say to them?

**E5.** Floating-point addition of the 100,000 values above differed by about 1.2e-07 on a total of about
1.1e+08. Express that as a relative error, and say how many significant figures of the total you can
trust.

**E6.** You run 30 seeds and report the best. Using 04-03's E6 arithmetic (the expected maximum of 30
standard normal draws is about 2.04), estimate how much better than the truth your reported number will
be, in MAE. Compare with the honest mean.

**E7.** A colleague's `requirements.txt` lists 12 packages without versions. If each has, say, 4 plausible
recent versions, how many distinct environments does that file describe?

### Coding

**E8.** Write `set_all_seeds(seed)` that seeds Python's `random`, numpy's legacy global generator, and
returns a `numpy.random.Generator`. Explain why the modern advice is to pass a generator explicitly
instead.

**E9.** Reproduce the split-seed versus model-seed comparison for a `Ridge` model instead of a forest.
Ridge has no randomness of its own - what does that predict, and does the measurement agree?

**E10.** Write `capture_environment()` returning a dict with python, key library versions, platform and
CPU count, and `compare_environments(a, b)` that prints only the differences. Test it by faking a version
change.

**E11.** Turn the experiment record into a function that appends one JSON line to a log file, then write
`load_experiments(path)` returning a DataFrame. Run three experiments and show the table sorted by result.

**E12.** Demonstrate that `n_jobs` can change the last decimals: fit the same forest with `n_jobs=1` and
`n_jobs=-1`, with the same `random_state`, and compare predictions exactly. Does it differ here?

### Interpretation

**E13.** A paper reports a new method beating the previous best by 0.3%, with no seed, no variance and no
environment. List what you would need before believing it, in order of how likely each is to explain the
0.3%.

**E14.** Your model's nightly retraining produces a score that drifts slowly downward over three months.
Nothing in the code changed. Name three explanations and the check for each.

### Debugging

**E15.** A result reproduces on your machine and not on a colleague's, and the difference is in the third
decimal. Where do you look, and where do you *not* bother looking?

**E16.** A result reproduces exactly on both machines, but the *conclusion* differs - your model wins and
theirs does not. What kind of difference produces that?

### Exam and interview reasoning

**E17.** "How do you make your work reproducible?" Answer in under a minute, in a way that shows you know
seeds are the easy part. Then: "we do not have time for all that - what is the minimum?"

### Transfer to a different situation

**E18.** You are handing a model to a production team. List what must travel with the fitted pipeline for
them to serve it correctly and to reproduce your evaluation later.

### Explain it to someone non-technical

**E19.** Explain in under 90 words why "it worked on my laptop last month" is not good enough, without
using the words seed, environment or version.

### Optional challenge

**E20.** Build a **reproducibility harness**: a function that runs an experiment, captures the record,
and - if a record for the same code, data and settings already exists in the log - re-runs it and asserts
the result matches. Demonstrate it catching a deliberate change (alter a hyperparameter, or remove a
seed). This is a regression test for a *result*, which is a thing very few projects have and almost all
of them want.

In [ ]:
# Your workspace. In memory: features, target, train_rows, test_rows, score,
# split_varies, model_varies, both_vary, record, experiment_record, entry.

## Mastery check

- [ ] Name the five levels and say what each fixes
- [ ] Measure how much of a reported number is seed choice, and which seed
- [ ] Explain why a seed does not survive a library upgrade
- [ ] Give a source of non-determinism a seed cannot touch, and say when it matters
- [ ] Write an experiment record with a baseline, a spread and an environment
- [ ] Diagnose a non-reproducing result in the order that costs least
- [ ] Say why a stable number is not the same as a representative one

## What should now feel instinctive

- Passing `random_state` to every estimator and splitter, as a reflex
- Reporting a spread, not a single number from one split
- Capturing library versions automatically rather than writing them down
- Treating "the notebook" as insufficient evidence of what was run
- Checking the environment first when a number moves

## Flashcards

| Front | Back |
|---|---|
| The five levels | Seeds, code, data, environment, hardware |
| What a seed fixes | Repeatability on this machine, today, with these libraries |
| Split seed vs model seed here | sd 0.0487 against 0.0094 - the split matters 5.2 times more |
| Stable versus representative | `random_state=0` gave 1.0032 where the average split gave 0.8809 |
| Why floats are not associative | Different summation orders differ in the last bit; threads change the order |
| Size of that effect here | About one part in 10^15 - irrelevant alone, compounding through branches |
| The level that actually breaks | Library versions, a year later, with no error message |
| Experiment record, minimum | why, code, data, split, settings, result **with spread**, baseline, environment |
| Diagnosis order | environment, data, code, seeds, arithmetic |
| Where a missing seed ranks | Fourth - it produces small noisy differences, not large ones |

## Module 04 complete

That is the workflow. Eight chapters, and every one of them was about something that happens **before**
or **around** the model rather than inside it:

frame the question so it has an answer · compute what free rules score · hold data out and read it once ·
respect entities and time when splitting · check the four ways the answer leaks in · make the
preprocessing decisions deliberately · put every fitted step inside a pipeline · record enough that
somebody can get your number back.

**None of it required knowing how a single model works**, and all of it decides whether the modelling was
worth doing. That is why this module sits before the algorithms rather than after them.

## Next

**Module 05 · Regression**, starting with **05-01 · The simplest possible predictor**. From here the
course fits models in earnest - least squares by hand, then multiple regression and what a coefficient
means, capacity and overfitting, regularisation, trees and boosting - and every one of them is evaluated
inside the workflow this module has built.

The first thing 05-01 does is compute a baseline.